<a href="https://colab.research.google.com/github/Sayak-coder/SIH_26086/blob/main/Ph_Nitrogen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade xee

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.9 MB/s eta 0:00:00


In [2]:
!pip install --upgrade xee

In [3]:
!pip install -u geemap


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [4]:
import ee

In [28]:
ee.Authenticate()
ee.Initialize(
    project='gen-lang-client-0960220624',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [6]:
import geemap

In [7]:
map=geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [9]:
roi=map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          -272.48291,
                          23.422928
                        ],
                        [
                          -272.48291,
                          24.287027
                        ],
                        [
                          -271.494141,
                          24.287027
                        ],
                        [
                          -271.494141,
                          23.422928
                        ],
                        [
                          -272.48291,
                          23.422928
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [10]:
def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

# Get the full geometry info to check its type and coordinates structure
roi_info = roi.getInfo()
geometry_type = roi_info['type']
coords = roi_info['coordinates']

if geometry_type == 'Point':
  # For a Point, coords is [lon, lat]
  lon = coords[0]
  lat = coords[1]
  fixed_coords = [wrap_lon(lon), lat]
  roi_fixed = ee.Geometry.Point(fixed_coords)
elif geometry_type == 'Polygon':
  # For a Polygon, coords is [[[lon, lat], ...], ...]
  fixed_coords = []
  for ring in coords: # Each ring is a list of [lon, lat] pairs
    fixed_ring = []
    for point in ring: # Each point is [lon, lat]
      fixed_ring.append([wrap_lon(point[0]), point[1]])
    fixed_coords.append(fixed_ring)
  roi_fixed = ee.Geometry.Polygon(fixed_coords)
else:
  # Handle other geometry types if necessary, or raise an error
  raise ValueError(f"Unsupported geometry type: {geometry_type}")

In [32]:
# ---------- Cropland mask (ESA WorldCover 2021, class 40 = cropland) ----------
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()  # single-image collection, never empty
cropland_mask = worldcover.select('Map').eq(40)

# ---------- Soil properties ----------
# pH in water, depth band b10 (10 cm), stored as pH * 10
ph_image = (ee.Image("OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02")
            .select(['b10'], ['ph'])
            .divide(10))

# Nitrogen from SoilGrids (cg/kg -> g/kg)
nitrogen_image = (ee.Image("projects/soilgrids-isric/nitrogen_mean")
                  .select(['nitrogen_5-15cm_mean'], ['nt'])
                  .divide(100))

soil_properties_image = ph_image.addBands(nitrogen_image)
soil_properties_masked = soil_properties_image.updateMask(cropland_mask)

# ---------- Sanity checks (remove once working) ----------
print("pH bands:      ", ph_image.bandNames().getInfo())
print("Nitrogen bands:", nitrogen_image.bandNames().getInfo())
print("Mask bands:    ", cropland_mask.bandNames().getInfo())
print("ROI type:      ", roi_fixed.type().getInfo())

# ---------- Zonal statistics ----------
soil_stats_collection = soil_properties_masked.reduceRegions(
    collection=roi_fixed,
    reducer=ee.Reducer.mean(),
    scale=250,          # native resolution of both soil datasets
    crs='EPSG:4326'
)

results = soil_stats_collection.getInfo()

print('\nRegion Properties (Mean in Croplands):')
for feature in results['features']:
    props = feature['properties']

    ph = props.get('ph')
    print(f"  pH: {ph:.2f}" if ph is not None
          else "  pH: No data or null (check cropland mask and ROI overlap)")

    nt = props.get('nt')
    print(f"  Nitrogen (g/kg): {nt:.2f}" if nt is not None
          else "  Nitrogen: No data or null (check cropland mask and ROI overlap)")


pH bands:       ['ph']
Nitrogen bands: ['nt']
Mask bands:     ['Map']
ROI type:       Polygon

Region Properties (Mean in Croplands):
  pH: 6.53
  Nitrogen (g/kg): 13.74
